# 03 The rerun

A dead end, kept, and the performance notes. The first idempotency test failed: the land task appended a drop to the landing table, and a rerun of the same day doubled it. The digest caught it before any total did, because a digest counts rows twice where a sum of amounts can hide in a monthly chart.

The fix is the rule now in AGENTS.md: no task appends without a key or a stated range. Every task overwrites the range it names.

In [1]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path

LAKE = Path("../data/lake/fixture/spark-warehouse")
def read(name):
    return pq.read_table(LAKE / name).to_pandas()

In [2]:
import hashlib

def digest(frame, cols):
    h = frame[cols].astype(str).agg("|".join, axis=1).map(lambda s: int(hashlib.sha256(s.encode()).hexdigest()[:15], 16))
    return len(frame), int(np.bitwise_xor.reduce(h.to_numpy())), int(h.sum() % (2**61 - 1))

silver = read("payments_silver")
day = silver[silver.event_ts.dt.date == silver.event_ts.dt.date.max()]
cols = ["txn_ref", "amount_cents", "event_ts"]
landed = pd.concat([silver[silver.event_ts.dt.date < day.event_ts.dt.date.min()], day])
appended_twice = pd.concat([landed, day])
overwritten = pd.concat([landed[~landed.txn_ref.isin(day.txn_ref)], day])
print("once      ", digest(landed, cols))
print("appended  ", digest(appended_twice, cols))
print("overwrite ", digest(overwritten, cols))
assert digest(landed, cols) == digest(overwritten, cols) and digest(landed, cols) != digest(appended_twice, cols)

once       (32353, 463985358613763865, 945694417980767914)


appended   (32366, 283195849841901430, 1124271769820684604)


overwrite  (32353, 463985358613763865, 945694417980767914)


XOR alone would have hidden the double: a row present twice cancels itself. The sum modulo a prime does not, which is why packages/transforms/digests.py keeps both.

## Performance notes

The state machines group by a hash bucket of the card or user, not by the entity, because millions of one user groups cost more in scheduling than in work. results/performance.json records the experiments (bucket count, file sizing, the pipeline's thread count) with their before and after timings.